In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.data import get_electrode_df, add_metadata_features
from src.models.decoding import run_decoding_analysis_single_electrode, get_ensemble_predictions

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
epochs_path = "outputs/epochs_preprocessed/EC260_epo.fif"
speech_responsive = "./EC260_results.csv"
outdir = "."

n_repeats = 10
window_size = 0.3  # seconds

In [ ]:
subject_name = re.findall("(EC[\d]+)_epo", epochs_path)[0]

In [ ]:
epochs_data = mne.read_epochs(epochs_path)
epochs_data.metadata = add_metadata_features(epochs_data.metadata)

In [ ]:
electrode_df = pd.read_csv(speech_responsive)
electrode_df

In [ ]:
global_min_sample = epochs_data.times.tolist().index(0)
global_max_sample = global_min_sample + int(window_size * epochs_data.info['sfreq'])

In [ ]:
train_scores, test_scores, outcomes, models = run_decoding_analysis_single_electrode(
    epochs={subject_name: epochs_data[epochs_data.metadata[epochs_data.metadata.resampled.isin((1, 6))].index]},
    global_min_sample=global_min_sample,
    global_max_sample=global_max_sample,
    electrode_df=electrode_df,
    filter_speech_responsive=True,
    target="acoustic",
    smoke_test=False,
    strategy="train-test",
    window_size=int(window_size * epochs_data.info['sfreq']),
    stride=5,  # not used -- not doing a sliding window analysis
)

In [ ]:
held_out_outcomes = {
    model_key: get_ensemble_predictions(model_key, models_i,
                                        epochs={subject_name: epochs_data[epochs_data.metadata[~epochs_data.metadata.resampled.isin((1, 6))].index]})
    for model_key, models_i in tqdm(models.items())
}

In [ ]:
train_scores_df = pd.concat(
    {key: pd.DataFrame(scores_i) for key, scores_i in train_scores.items()},
    names=["subject", "electrode_idx", "phoneme_pair", "smin", "smax", "fold"])
train_scores_df

In [ ]:
scores_df = pd.concat(
    {key: pd.DataFrame(scores_i) for key, scores_i in test_scores.items()},
    names=["subject", "electrode_idx", "phoneme_pair", "smin", "smax", "fold"])
scores_df

In [ ]:
# average over folds/repeats
avg_scores_df = scores_df.groupby(["subject", "electrode_idx", "phoneme_pair", "smin", "smax"]).mean().sort_values("roc_auc", ascending=False)
avg_scores_df

## Estimate significance threshold

The expected ROC-AUC for a random classifier is distributed $\text{Beta}(N_+, N_-)$, where $N_+$ is the number of positive samples and $N_-$ is the number of negative samples. We can use this to estimate a significance threshold for our ROC-AUC scores.

In [ ]:
data_sample = next(iter(outcomes.values())).query("fold == 0")  # don't over-estimate sample counts by double-counting across folds
class_counts = data_sample.groupby("decoder_target").size().tolist()
print(f"Class counts: {class_counts}")

from scipy.stats import beta
confidence_threshold = 0.95
alpha = 1 - confidence_threshold
significance_threshold = beta.ppf(1 - alpha, *class_counts)
print(f"Significance threshold: {significance_threshold:.3f}")

In [ ]:
import seaborn as sns
g = sns.displot(data=avg_scores_df,
                x="roc_auc", aspect=2, height=4, kind="kde", fill=True, color="blue")
g.ax.axvline(significance_threshold, color="red", linestyle="--", label="Significance threshold")

## Declare results

In [ ]:
avg_scores_df["A"] = False
avg_scores_df.loc[avg_scores_df.roc_auc > significance_threshold, "A"] = True
avg_scores_df.A.mean()

In [ ]:
avg_scores_df.to_csv(f"{outdir}/{subject_name}_results.csv")

In [ ]:
torch.save({
    "train_scores": train_scores_df,
    "test_scores": scores_df,
    "avg_scores": avg_scores_df,
    "outcomes": outcomes,
    "held_out_outcomes": held_out_outcomes,
    "models": models
}, f"{outdir}/{subject_name}_decoders.pt")